# Phase 4 - MLFlow Experiment Tracking

MLFlow UI runs at http://127.0.0.1:5000

run mlflow ui

In [13]:
import mlflow
import os
import joblib
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import json
from sklearn.metrics import accuracy_score, f1_score, recall_score

# Point MLFlow to the right location
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

print('MLFlow Version: ',mlflow.__version__)
print('Tracking URI: ',mlflow.get_tracking_uri())

MLFlow Version:  3.16.0
Tracking URI:  sqlite:///../mlflow.db


In [4]:
# Set the Experiment

mlflow.set_experiment("healthcare-risk-classification")
print('Experiment Created ✅')

2026/09/28 05:01:39 INFO mlflow.tracking.fluent: Experiment with name 'healthcare-risk-classification' does not exist. Creating a new experiment.


Experiment Created ✅


In [7]:
risk_rf_model = joblib.load("../models/risk_model.joblib")
claim_rf_model = joblib.load("../models/claim_model.joblib")
print("Models loaded ✅")
print("risk_model: ",type(risk_rf_model))
print("claim_model: ",type(claim_rf_model))

Models loaded ✅
risk_model:  <class 'sklearn.pipeline.Pipeline'>
claim_model:  <class 'sklearn.pipeline.Pipeline'>


In [11]:
# reading data

df = pd.read_csv("../outputs/model_table.csv",parse_dates=['registration_date','visit_date','billing_date'])
print("Dataset loaded ✅")
print("Shape of dataset: ", df.shape)
df.head(3)

Dataset loaded ✅
Shape of dataset:  (25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,2,15,F,Mumbai,CareOne,0,2025-12-27,8,2026-01-01,General,...,0,2,1,5,4,21.12,0.256876,1,3,0
1,12,3,M,Bangalore,CareOne,0,2025-08-13,65,2026-01-01,ICU,...,2,2,1,141,8,23.75,0.256876,1,3,1
2,129,44,M,Pune,MediCareX,1,2025-07-20,651,2026-01-01,ICU,...,2,1,0,165,3,32.46,0.242556,1,3,1


In [16]:
# loading features

with open('../outputs/feature_schema.json',"r") as f:
    schema = json.load(f)

risk_feature = schema["risk_model_features"]
claim_feature = schema["claim_model_features"]
risk_target = schema['risk_target']
claim_target = schema['claim_target']

print('Schema loaded ✅')
print(f"Risk features: {len(risk_feature)}")
print(f"Claim features: {len(claim_feature)}")


Schema loaded ✅
Risk features: 14
Claim features: 18


In [17]:
# create separate datasets

risk_df = df.copy()
claim_df = df.copy()

In [21]:
# Time based split

risk_df = risk_df.sort_values(by= ['visit_date']).reset_index(drop=True)

split_idx = int(len(risk_df) * 0.8)

risk_train = risk_df.iloc[:split_idx].copy()
risk_test = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_feature]
X_test_risk = risk_test[risk_feature]

y_train_risk = risk_train[risk_target]
y_test_risk = risk_test[risk_target]

print("Train shape: ", X_train_risk.shape)
print("Test shape: ", X_test_risk.shape)
print("Train Period: ", risk_train["visit_date"].min().date(), "->", risk_train["visit_date"].max().date())
print("Test Period: ", risk_test["visit_date"].min().date(), "->", risk_test["visit_date"].max().date())

Train shape:  (20000, 14)
Test shape:  (5000, 14)
Train Period:  2025-01-21 -> 2026-01-02
Test Period:  2026-01-02 -> 2026-01-20


In [22]:
claim_df = claim_df.sort_values("billing_date").reset_index(drop=True)
split_idx = int(len(claim_df)*0.8)

claim_train = claim_df.iloc[:split_idx].copy()
claim_test = claim_df.iloc[split_idx:].copy()

X_train_claim = claim_train[claim_feature]
X_test_claim = claim_test[claim_feature]

y_train_claim = claim_train[claim_target]
y_test_claim = claim_test[claim_target]

print("Train shape:", X_train_claim.shape)
print("Test shape :", X_test_claim.shape)
print("Train period:", claim_train["billing_date"].min().date(),
      "→", claim_train["billing_date"].max().date())
print("Test period :", claim_test["billing_date"].min().date(),
      "→", claim_test["billing_date"].max().date())

Train shape: (20000, 18)
Test shape : (5000, 18)
Train period: 2025-01-28 → 2026-01-17
Test period : 2026-01-17 → 2026-01-20


In [25]:
# log risk model

with mlflow.start_run(run_name = "RandomForest - Risk") as run:
    mlflow.log_params(risk_rf_model.get_params()
        # {
        # "model": "RandomForestClassifier",
        # "n_estimators": 200,
        # "max_depth": 8,
        # "min_samples_split": 20,
        # "class_weight": "balanced_subsample",
        # "evaluation_data": "risk_test_only",
        # "split_strategy": "time_based_80_20",
        # "split_column": "visit_date"
        # }
    )

    # Metrics
    pred_risk = risk_rf_model.predict(X_test_risk)
    acc_risk = accuracy_score(y_test_risk,pred_risk)
    f1_risk = f1_score(y_test_risk,pred_risk, average="weighted")
    high_recall = recall_score(y_test_risk,pred_risk, labels=['High'], average=None)[0]

    # logging the metrics

    mlflow.log_metric("accuracy", acc_risk)
    mlflow.log_metric("weighted_f1", f1_risk)
    mlflow.log_metric("high_risk_recall", high_recall)

    #logging the model
    mlflow.sklearn.log_model(
        sk_model= risk_rf_model,
        name= "model",
        skops_trusted_types= ["numpy.dtype"]
    )

    risk_run_id = run.info.run_id

    print("RandomForest Risk Model logged ✅")
    print(f"   Accuracy          : {acc_risk: .4f}")
    print(f"   Weighted F1       : {f1_risk: .4f}")
    print(f"   High Risk Recall  : {high_recall: .4f}")
    print(f"   Run ID          : {risk_run_id}")

2026/09/28 23:31:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RandomForest Risk Model logged ✅
   Accuracy          :  0.9350
   Weighted F1       :  0.9350
   High Risk Recall  :  0.9297
   Run ID          : 53c69e0621394a71a871f5d7159e2106


In [27]:
with mlflow.start_run(run_name="RandomForest-Claim") as run:

    mlflow.log_params({
        "model": "RandomForestClassifier",
        "n_estimators": 250,
        "max_depth": 14,
        "min_samples_split": 8,
        "class_weight": "balanced",
        "evaluation_data": "claim_test_only",
        "split_strategy": "time_based_80_20",
        "split_column": "billing_date"
    })

    pred_claim = claim_rf_model.predict(X_test_claim)

    acc_claim = accuracy_score(y_test_claim, pred_claim)
    f1_claim = f1_score(y_test_claim, pred_claim, average="weighted")
    rejected_recall = recall_score(
        y_test_claim, pred_claim, labels=["Rejected"], average=None
    )[0]

    mlflow.log_metric("accuracy", acc_claim)
    mlflow.log_metric("weighted_f1", f1_claim)
    mlflow.log_metric("rejected_recall", rejected_recall)

    mlflow.sklearn.log_model(
        sk_model=claim_rf_model,
        name="model",
        skops_trusted_types= ["numpy.dtype"]
    )

    claim_run_id = run.info.run_id

    print("RandomForest Claim Model logged ✓")
    print(f"  Accuracy         : {acc_claim:.4f}")
    print(f"  Weighted F1      : {f1_claim:.4f}")
    print(f"  Rejected Recall  : {rejected_recall:.4f}")
    print(f"  Run ID           : {claim_run_id}")

2026/09/28 23:38:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RandomForest Claim Model logged ✓
  Accuracy         : 0.5000
  Weighted F1      : 0.4879
  Rejected Recall  : 0.3010
  Run ID           : 7e5d288ceaf144ee8fd0841c947e52c1


# Register Model in MLFlow

In [30]:
from mlflow import register_model

register_model_name = 'HealthRiskRFModel'

model_uri = f"runs:/{risk_run_id}/model"

result = register_model(
    model_uri= model_uri,
    name= register_model_name
)

print("Register Model Name: ", result.name)
print("Register Model Version:", result.version)

Registered model 'HealthRiskRFModel' already exists. Creating a new version of this model...
2026/09/29 05:29:06 WARNING mlflow.tracking._model_registry.fluent: Run with id 53c69e0621394a71a871f5d7159e2106 has no artifacts at artifact path 'model', registering model based on models:/m-c164b95997a74fb98ad89117c496a454 instead


Register Model Name:  HealthRiskRFModel
Register Model Version: 2


Created version '2' of model 'HealthRiskRFModel'.


In [31]:
risk_model_version = result.version

print("Risk Model Version Saved = ", result.version)

Risk Model Version Saved =  2


In [32]:
from mlflow.tracking import MlflowClient

client = MlflowClient()


In [33]:
# Move model to staging
client.transition_model_version_stage(
    name = register_model_name,
    version = risk_model_version,
    stage = "Staging"
)

print(f"model {register_model_name} version {risk_model_version} moved to Staging")

model HealthRiskRFModel version 2 moved to Staging


In [34]:
# check if model is good to push to production
if acc_risk >= 0.55 and high_recall >= 0.70:
    print("Risk Model is eligible for production Promotion ✅")
else:
    print("Risk Model is not eligible")

Risk Model is eligible for production Promotion ✅


In [36]:
client.transition_model_version_stage(
    name = register_model_name,
    version = risk_model_version,
    stage = "Production",
    archive_existing_versions = True
)

print(f"model {register_model_name} version {risk_model_version} moved to Production")

model HealthRiskRFModel version 2 moved to Production


In [38]:
# load the production model
import mlflow.sklearn

production_risk_model = mlflow.sklearn.load_model(
    model_uri=f"models:/{register_model_name}/Production"
)
print('Model loaded')

Model loaded


In [39]:
latest_versions = client.get_latest_versions(
    name=register_model_name,
    stages= ['Production']
)

production_version = latest_versions[0].version if latest_versions else None

print("Current Production version:",production_version)

Current Production version: 2


In [40]:
# prediction from Production model
pred_risk = production_risk_model.predict(X_test_risk.head(5))
pred_risk

array(['Low', 'High', 'Low', 'Medium', 'Medium'], dtype=object)

In [50]:
# log prediction with model version
import hashlib
from datetime import datetime

def hash_input(payload: dict) -> str:
    payload_str = json.dumps(payload, sort_keys=True)
    return hashlib.sha256(payload_str.encode()).hexdigest()

In [48]:
input_payload = {
    "age": 52,
    "gender": "M",                        
    "city": "Bangalore",
    "insurance_provider": "CareOne",       
    "chronic_flag": 1,
    "department": "Cardiology",
    "visit_type": "ER",                    
    "doctor_id": 101,
    "length_of_stay_hours": 48,
    "days_since_registration": 300,
    "visit_frequency": 4,
    "avg_los_per_patient": 36.5,
    "visit_month": 3,
    "visit_dayofweek": 2
}

In [49]:
input_hash = hash_input(input_payload)
print(input_hash)

393af61ee47c8f37fa64b93931d86baa4a2690fc9a9ecf27987b660d34e5c8bd


In [53]:
# Step 1: Define logs directory
BASE_DIR = os.getcwd()  # or project root if running from notebooks
LOG_DIR = os.path.join(BASE_DIR, "logs")

# Step 2: Create logs folder if not exists
os.makedirs(LOG_DIR, exist_ok=True)

# Step 3: Define log file path
LOG_FILE = os.path.join(LOG_DIR, "predictions.log")

# Step 4:
prediction_log = {
    "timestamp": datetime.now().isoformat(),
    "model_name": register_model_name,
    "model_version": production_version,
    "input_hash": input_hash,
    "prediction": str(pred_risk[0])
}

with open(LOG_FILE, "a", encoding="utf-8") as f:
    f.write(json.dumps(prediction_log) + "\n")

print("Prediction logged with model_version ✓")

Prediction logged with model_version ✓
